# NB3 — depth 재-anchor (global a) **[GPU 권장, CPU 가능]**
1. `extract_pool_fit.py` — 전 프레임 LiDAR 점쌍 pool → 단일 전역 `a` (RANSAC+MAD, RadarCam Const). **세션마다 새로 적합** (4/20 = 0.486)
2. `phase4_reanchor.py` — `z = a_global·s + b_frame` 로 per-frame flicker 제거 + tail 제거 + Poisson refine. `--dry` 미리보기 → `--apply` 덮어쓰기 (`.bak` 백업)

입력: NB2 `01_depth/work/{video,out}` · 산출: `work/out/*_person_metric.npy`(재-anchor), `work/analysis/{global_const_fit,phase4_report}.json`

In [ ]:
#@title 0. 설정 — 세션마다 여기만 바꾸면 됨
ROOT    = "/content/drive/MyDrive/walking_0729"   #@param {type:"string"}
SESSION = "0729_walk_3kmh"                        #@param {type:"string"}

from google.colab import drive
drive.mount("/content/drive")
import os, sys, importlib
os.environ["WALK_ROOT"] = ROOT
os.environ["WALK_SESSION"] = SESSION
sys.path.insert(0, f"{ROOT}/scripts")
import walkcfg; importlib.reload(walkcfg)
assert os.path.isdir(f"{ROOT}/scripts"), f"{ROOT}/scripts 없음 — colab_kit 업로드 위치 확인"
print("세션 폴더:", walkcfg.SESS)

In [ ]:
#@title 1. 전역 a 적합
!cd "{ROOT}/scripts" && python extract_pool_fit.py
import json
g = json.load(open(walkcfg.GLOBAL_JSON))
print("\nchosen_global_a =", g["chosen_global_a"], "  (4/20 = 0.486)")

In [ ]:
#@title 2. 재-anchor 미리보기 (--dry, 파일 안 바뀜)
!cd "{ROOT}/scripts" && python phase4_reanchor.py --dry

In [ ]:
#@title 3. 적용 (--apply, 원본은 .bak 백업)
!cd "{ROOT}/scripts" && python phase4_reanchor.py --apply
print(open(f"{walkcfg.DEPTH_BASE}/work/analysis/phase4_report.json").read()[:3000])

In [ ]:
#@title 4. (선택) 결과 확인 후 .bak 삭제 — 드라이브 용량 회수 (~11MB/프레임)
DELETE_BAK = False  #@param {type:"boolean"}
import glob
baks = glob.glob(f"{walkcfg.DEPTH_SEQ}/*.bak")
print(len(baks), ".bak 파일")
if DELETE_BAK:
    for b in baks: os.remove(b)
    print("삭제 완료")

## 시각화 (NB3)

In [ ]:
#@title 시각화 — NB3 결과 확인 (viz/NB3/  + viz/videos/)
FIDS = "5660 5750"   #@param {type:"string"}
N_EVEN = 0           #@param {type:"integer"}
# 셀 0 을 안 돌렸어도 혼자 동작하도록 (드라이브 마운트 + 세션 설정)
import os, sys, importlib
if "walkcfg" not in globals():
    ROOT = "/content/drive/MyDrive/walking_0729"   # 셀 0 과 같게
    SESSION = "0729_full_3kmh"                     # 셀 0 과 같게
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["WALK_ROOT"] = ROOT; os.environ["WALK_SESSION"] = SESSION
    sys.path.insert(0, f"{ROOT}/scripts"); import walkcfg
print("세션:", walkcfg.SESS)
MAKE_VIDEO = True    #@param {type:"boolean"}
FPS = 15             #@param {type:"number"}
!pip -q install imageio-ffmpeg
!cd "{ROOT}/scripts" && python viz_stages.py --nb 3 --fids {FIDS} --n {N_EVEN}
if MAKE_VIDEO:
    !cd "{ROOT}/scripts" && python viz_video.py --nb 3 --fps {FPS}
from IPython.display import Image, display
import glob, os
for p in sorted(glob.glob(f"{walkcfg.SESS}/viz/NB3/*.png")):
    print(os.path.basename(p)); display(Image(p, width=1100))
from IPython.display import Video
_mp4 = f"{walkcfg.SESS}/viz/videos/NB3_reanchor.mp4"
if os.path.exists(_mp4):
    print("영상:", _mp4, f"{os.path.getsize(_mp4)/1e6:.1f} MB  (노트북에 넣어 재생 — 확인 후 출력 지우기 권장)")
    display(Video(_mp4, embed=True, width=1100))